# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amey05081999/Flyrank-Internship-Amey-Naik/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**1) What one row means:** One row in `fact_content_daily_performance` is one **content item × client × report date** observation.

**2) Tables used:** I use `fact_content_daily_performance` for daily search-performance signals and `dim_content` only when content metadata is needed. For this Week 3 slice, the five-feature frame is built from the daily fact so that feature windows can be aligned cleanly to the decision date.

**3) Time window:** Features use the March 2026 observation window, ending at the decision moment of **2026-03-31**. The outcome is measured in the following 30-day window, **2026-04-01 through 2026-04-30**. I do not develop label logic from the `_sample` table because it is June 2026, the natural final outcome window.

**4) What I predict/rank:** I predict a binary **next-30-day search-impression decline**: April impressions are more than 20% below March impressions. The model output is intended as a decision-support score for which content items deserve earlier search-performance attention.

**5) Deliberately excluded:** `gsc_avg_position` is not used as a raw model feature when it represents no-data values; rows with unavailable GSC/search data are excluded from the modeling frame rather than treating zero as rank 0. I also exclude all client/content/query identifiers from model features because they are grouping/joining context, not signals.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Features** | `march_impressions`, `march_clicks`, `march_ctr`, `march_avg_position`, `march_sessions` | All are measurable by the decision moment and describe search/traffic performance available during March. |
| **Label / proxy** | `next30_impression_decline` | Derived only from the future April outcome; it is what the model predicts and therefore cannot be a feature. |
| **Context** | `client_id_hash`, `content_id_hash`, `report_date`, `month` | Needed to define the grain, join, filter, split, and time windows; not signals for the model. |
| **Excluded** | `trend_direction`, `trend_pct`, any April outcome metric, and raw identifiers | They either encode the outcome/future window directly or are identifiers. Using them would leak future information or memorize entities. |

**Availability rule:** GA4 fields are used only where `ga4_data_available IS TRUE`. The warehouse documentation warns that GA4 can be zero-filled before a client's GA4 start and can also be NULL; therefore `= FALSE` / `NOT flag` is not an adequate availability test.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries

The next three SQL cells are the **three verification queries** for this contract. They use the mid-panel March 2026 partition and keep the final June month sealed.

### Query 1 — Grain

If the declared grain is correct, no `(report_date, client_id_hash, content_id_hash)` combination should occur more than once.

In [26]:
import os, duckdb, pandas as pd

con = duckdb.connect()
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN as a Colab Secret/environment variable before running this notebook.")
con.execute("CREATE SECRET IF NOT EXISTS (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

q1 = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS rows_at_key
FROM read_parquet('{MARCH}')
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Duplicate grain keys returned:", len(q1))
display(q1)
assert q1.empty, "Declared daily grain is violated: duplicate report_date × client × content keys found."


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain keys returned: 0


,report_date,client_hash_id,content_hash_id,rows_at_key


### Query 2 — March slice row count and date span

In [27]:
# Verification query 2 — row count + date span
q2 = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet('{MARCH}')
""").df()

display(q2)

assert str(q2.loc[0, "min_report_date"])[:10] == "2026-03-01"
assert str(q2.loc[0, "max_report_date"])[:10] == "2026-03-31"

,march_rows,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — GA4 availability using `IS TRUE`

The warehouse has three-valued availability flags. This check deliberately uses **`IS TRUE`**, not `= TRUE`, and reports how many March rows have usable GA4 data.

In [28]:
# Verification query 3 — availability; use IS TRUE
q3 = con.sql(f"""
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_rows_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_rows_not_available_or_null
FROM read_parquet('{MARCH}')
""").df()

display(q3)

assert int(q3.loc[0, "ga4_rows_available"]) <= int(q3.loc[0, "march_rows"])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,ga4_rows_available,ga4_rows_not_available_or_null
0,9841378,413966,9427412


## 3b. Five-feature frame

The feature frame is built at **content × client** grain using March only. Every feature is knowable at the 2026-03-31 decision moment.

| Feature | Available when? |
|---|---|
| `march_impressions` | Knowable at the decision moment because it is the sum of GSC impressions observed during March, with no April data used. |
| `march_clicks` | Knowable at the decision moment because GSC clicks through 2026-03-31 have already been observed. |
| `march_ctr` | Knowable at the decision moment because it is calculated only from March clicks and March impressions. |
| `march_avg_position` | Knowable at the decision moment because it is calculated from March GSC position observations; unavailable/zero-position rows are treated as missing rather than rank 0. |
| `march_sessions` | Knowable at the decision moment because GA4 sessions are aggregated only where `ga4_data_available IS TRUE` during March. |

In [30]:
# Five-feature frame + future label.
# This is intentionally a separate extraction step, not one of the three contract-verification queries above.

feature_label_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,
        SUM(
            CASE
                WHEN gsc_avg_position IS NOT NULL AND gsc_avg_position > 0
                THEN gsc_avg_position * COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) / NULLIF(
            SUM(
                CASE
                    WHEN gsc_avg_position IS NOT NULL AND gsc_avg_position > 0
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ), 0
        ) AS march_avg_position,
        SUM(
            CASE WHEN ga4_data_available IS TRUE
                 THEN COALESCE(ga4_sessions, 0)
                 ELSE 0 END
        ) AS march_sessions
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1,2
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS april_impressions
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')
    GROUP BY 1,2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    100.0 * m.march_clicks / NULLIF(m.march_impressions, 0) AS march_ctr,
    m.march_avg_position,
    m.march_sessions,
    CASE
        WHEN a.april_impressions IS NULL OR m.march_impressions = 0 THEN NULL
        WHEN a.april_impressions < 0.8 * m.march_impressions THEN 1
        ELSE 0
    END AS next30_impression_decline
FROM march m
LEFT JOIN april a USING (client_hash_id, content_hash_id)
WHERE m.march_impressions > 0
"""

feature_frame = con.sql(feature_label_sql).df()

FEATURES = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_sessions",
]
LABEL = "next30_impression_decline"

display(feature_frame[FEATURES + [LABEL]].head(10))
print("Feature-frame rows:", len(feature_frame))
print("Rows with observed future label:", feature_frame[LABEL].notna().sum())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_impressions,march_clicks,march_ctr,march_avg_position,march_sessions,next30_impression_decline
0,1423.0,1.0,0.070274,16.168658,0.0,0
1,1.0,0.0,0.000000,9.000000,0.0,0
2,208.0,0.0,0.000000,32.438272,0.0,0
3,607.0,0.0,0.000000,30.583196,0.0,1
4,267.0,0.0,0.000000,21.071161,0.0,0
5,1489.0,0.0,0.000000,48.435191,0.0,1
6,2091.0,4.0,0.191296,9.396939,0.0,0
7,10239.0,31.0,0.302764,14.609728,0.0,0
8,14061.0,32.0,0.227580,4.593343,0.0,1
9,12022.0,7.0,0.058227,7.827649,0.0,1


Feature-frame rows: 176738
Rows with observed future label: 176737


## 3c. Deliberate leakage trap

**Trap:** I add the label itself as a feature on purpose. This is not a legitimate modeling feature. It should make a quick classifier look almost perfect because the model is being handed the answer.

I then remove the leaked column and report the honest score from the five-feature frame.

The experiment uses a deterministic train/test split so the comparison is reproducible. No client/content identifiers are model features.

In [31]:
# Deliberate label leakage experiment.
# The leaked feature is the future label itself. It is added only to demonstrate the failure mode,
# then deleted before the honest score is reported.

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

model_df = feature_frame.dropna(subset=[LABEL]).copy()

# Keep only the five honest features for the real model.
X_honest = model_df[FEATURES].copy()
y = model_df[LABEL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X_honest, y, test_size=0.25, random_state=42, stratify=y
)

def fit_score(Xtr, Xte):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        LogisticRegression(max_iter=1000)
    )
    model.fit(Xtr, y_train)
    proba = model.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    }

honest_score = fit_score(X_train, X_test)

# ONE label-derived column, intentionally leaked.
X_leak = X_honest.copy()
X_leak["label_leak"] = y.values

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leak, y, test_size=0.25, random_state=42, stratify=y
)

leak_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(max_iter=1000)
)
leak_model.fit(Xl_train, yl_train)
leak_proba = leak_model.predict_proba(Xl_test)[:, 1]
leak_pred = (leak_proba >= 0.5).astype(int)

leak_score = {
    "accuracy": accuracy_score(yl_test, leak_pred),
    "roc_auc": roc_auc_score(yl_test, leak_proba),
}

scores = pd.DataFrame(
    [leak_score, honest_score],
    index=["WITH deliberate label leak", "HONEST — leak removed"]
)

display(scores)

print(
    "Leakage lesson: the leaked column is the answer, so its near-perfect score is not evidence "
    "of predictive power. It must be deleted before any real evaluation or model comparison."
)

# Keep only the honest five-feature frame for downstream work.
X_honest = model_df[FEATURES].copy()
print("Final model feature set:", FEATURES)
assert "label_leak" not in X_honest.columns


,accuracy,roc_auc
WITH deliberate label leak,1.000000,1.000000
HONEST — leak removed,0.556705,0.590879


Leakage lesson: the leaked column is the answer, so its near-perfect score is not evidence of predictive power. It must be deleted before any real evaluation or model comparison.
Final model feature set: ['march_impressions', 'march_clicks', 'march_ctr', 'march_avg_position', 'march_sessions']


## 4. Data limits

**Named limitation — unbalanced history / measurement availability.** The warehouse is an unbalanced panel: clients have different GSC and GA4 history depths, and GA4 availability is explicitly flagged. Therefore this slice can support **decision-support comparisons among content items with observed March search data**, but it cannot be interpreted as a complete picture of all content or as evidence that missing GA4 observations mean zero engagement.

A second practical limit is that March is a development month. June 2026 is intentionally kept sealed as the natural final outcome month.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.